# Unit 5 — Spatio-Temporal GNNs: earn the complexity, interrogate the attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bgalon/geo-graph-learning/blob/main/unit-5-gnn/geoai-graph-unit5.ipynb)

**Capstone.** Unit 3 *measured* a gap: forecasting one freeway sensor from its
own past, a smart model (SARIMA) stops beating a dumb historical-average floor
past a short **breakeven horizon** — because a univariate model is blind to the
**upstream neighbor** whose congestion will arrive in minutes. That blind spot is
the **spatial gap**. This unit closes it with a **spatio-temporal graph neural
network** — and then asks the harder question: when the model is right, can you
*read why*, and is that a defensible spatial story or a coincidence?

**What you'll build (live).**
1. Load **METR-LA** (207 LA freeway sensors) as a spatio-temporal graph and find
   a real congestion **shockwave** propagating upstream → downstream.
2. **Live-train a T-GCN** (a GCN inside a GRU) and watch the loss fall.
3. Race it against **Unit 3's SARIMA + historical-average floor** across
   15/30/60-min horizons — *did the GNN push the breakeven horizon outward?*
4. **Load a pre-trained A3T-GCN** and read its **attention** on the shockwave —
   honestly: it is attention over **time steps**, *not* over neighbor nodes, so
   we pair it with Unit 3's **spatial** cross-correlation for the "which upstream
   node" story. A temporal-attention heatmap is **not** spatial causality.
5. Re-frame the **bus corridor** (Unit 3's ~10 segments) as a graph and show a
   GNN *trains* on it — the on-ramp to your practice task.

**Prerequisites.** PyTorch basics (tensors, an optimizer loop); from **Unit 1**
the graph/adjacency/CRS mental model; from **Unit 3** the breakeven horizon, the
MAE/RMSE/MAPE protocol, the upstream cross-correlation result, and METR-LA;
from **Unit 4** the time-varying-edge-weight framing (helpful, not required).

**Data & attribution.** METR-LA is documented by Li, Yu, Shahabi & Liu (2018),
*DCRNN* ([arXiv:1707.01926](https://arxiv.org/abs/1707.01926)) and is bundled in
PyTorch Geometric Temporal (auto-downloads — nothing to pre-stage). Models:
T-GCN, Zhao et al. (2019) ([arXiv:1811.05320](https://arxiv.org/abs/1811.05320));
A3T-GCN, Bai et al. (2021) ([arXiv:2006.11583](https://arxiv.org/abs/2006.11583),
DOI [10.3390/ijgi10070485](https://doi.org/10.3390/ijgi10070485)).

> Rights reserved (not open-CC). AI-assisted material; see the course NOTICE.


## Setup

On **Colab** this installs the Unit 5 stack (`torch_geometric` +
`torch-geometric-temporal`; PyTorch is whatever Colab ships — we never pin it).
Locally with `uv` it is a no-op. **First run takes ~1–2 min** — the PyG install
is the one slow step of this unit; let it finish before running anything below.


In [ ]:
# --- Setup: fetch the course helper and install this unit's requirements -----
# On Colab: pip-installs requirements/unit-5.txt (no torch pin; scatter/sparse
# intentionally omitted). Locally: no-op (env already set up with `uv`).
import os, sys, subprocess, urllib.request

SETUP_URL = (
    "https://raw.githubusercontent.com/"
    "bgalon/geo-graph-learning/main/setup_colab.py"
)
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("setup_colab.py"):
        urllib.request.urlretrieve(SETUP_URL, "setup_colab.py")
    import setup_colab
    setup_colab.setup_unit("unit-5")
else:
    print("Local environment detected — skipping Colab install "
          "(assuming `uv sync --extra unit-5`).")

## Smoke test — fail fast, fail clean

Bare import of every major library + one trivial call each, **before any teaching
content**, so a broken environment surfaces here (pointing at
`KNOWN_ISSUES.md`) and not 30 minutes in. Optional/fragile extensions
(`torch_scatter`/`torch_sparse`) are import-guarded — their absence is a
**non-fatal** warning (PyG ≥2.4 uses native scatter; we don't need them).


In [ ]:
# --- Smoke test --------------------------------------------------------------
# CRITICAL (IPython-9 gotcha): we capture any failure, then re-raise OUTSIDE the
# except block. Re-raising *inside* the handler triggers a chained-traceback mess
# on Colab's IPython 9.x. So: set a flag in the except, raise after it.
_smoke_error = None
try:
    import torch
    import torch_geometric
    from torch_geometric.nn import GCNConv
    import torch_geometric_temporal
    from torch_geometric_temporal.nn.recurrent import TGCN, A3TGCN
    import sklearn
    from sklearn.metrics import mean_absolute_error
    import matplotlib
    import matplotlib.pyplot as plt
    import folium
    import numpy as np

    # One trivial call each:
    _ = torch.tensor([1.0, 2.0]).mean().item()
    _ = GCNConv(2, 2)                                   # constructs a layer
    _ = mean_absolute_error([1.0, 2.0], [1.1, 1.9])    # sklearn metric works
    _ = folium.Map(location=[34.05, -118.24], zoom_start=10)  # 1-point map
    _fig = plt.figure(); plt.close(_fig)               # matplotlib backend OK
except Exception as exc:                                # noqa: BLE001
    _smoke_error = exc

# --- Optional/fragile extensions: import-guarded, NON-fatal ------------------
_HAVE_SCATTER = False
try:
    import torch_scatter  # noqa: F401
    _HAVE_SCATTER = True
except Exception:                                       # noqa: BLE001
    pass  # expected & fine — PyG ≥2.4 uses native torch scatter.

# --- Raise OUTSIDE the except (clean banner; IPython-9 safe) -----------------
if _smoke_error is not None:
    print("=" * 64)
    print("SMOKE TEST FAILED — environment is not ready.")
    print(f"  {type(_smoke_error).__name__}: {_smoke_error}")
    print("  See unit-5-gnn/KNOWN_ISSUES.md (torch/PyG version conflicts,")
    print("  CUDA-absent path, METRLADatasetLoader/gdown network failures).")
    print("=" * 64)
    raise _smoke_error from None

# --- Success banner + GPU presence (the room needs to know before live-train) -
_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("✓ smoke test passed")
print(f"  torch            {torch.__version__}")
print(f"  torch_geometric  {torch_geometric.__version__}")
print(f"  PyG-Temporal     {torch_geometric_temporal.__version__}")
print(f"  device           {_DEVICE.upper()}"
      + ("  (GPU present — live-train will be fast)"
         if _DEVICE == "cuda"
         else "  (NO GPU — flip USE_PRETRAINED=True below if live-train is slow)"))
print(f"  torch_scatter    {'present' if _HAVE_SCATTER else 'absent (fine — native scatter)'}")

## Helpers — fetch-with-fallback, masked metrics, distribution + loss plots

Defined once, reused throughout. The fetch helpers print **which path was taken**
(primary vs. fallback) — never silent — and **cache to `/content`** so re-runs
are instant. `plot_error_distributions` draws **box + histogram + ECDF** (used at
every error comparison, per the show-don't-print rule).


In [ ]:
# --- Config: data/checkpoint URLs (no <PASTE_URL> placeholders on the data path)
# Checkpoints ship via the course Google Drive (gdown) with a small HTTP mirror
# as a hosted fallback. METR-LA itself auto-downloads via the PyG-Temporal loader.
CACHE_DIR = "/content" if IN_COLAB else "."

# gdown file ids (course Drive) + HTTP mirror (public-repo release asset).
CKPT = {
    "tgcn_metrla":   {"gdrive_id": "<GDRIVE_ID_tgcn_metrla>",
                      "http": "https://github.com/bgalon/geo-graph-learning/releases/download/u5-ckpts/tgcn_metrla.pt"},
    "a3tgcn_metrla": {"gdrive_id": "<GDRIVE_ID_a3tgcn_metrla>",
                      "http": "https://github.com/bgalon/geo-graph-learning/releases/download/u5-ckpts/a3tgcn_metrla.pt"},
}
# Tiny pre-built bus graph + a short cached speed slice (Unit 3 corridor).
BUS = {"gdrive_id": "<GDRIVE_ID_bus_graph>",
       "http": "https://github.com/bgalon/geo-graph-learning/releases/download/u5-data/bus_corridor_u5.npz"}

# Checkpoint hygiene: the .pt files are state_dicts trained against the PINNED
# PyG / PyG-Temporal versions recorded next to each file on Drive. If your
# installed versions differ and load_state_dict complains, re-export (instructor)
# — see KNOWN_ISSUES.md.

In [ ]:
# --- fetch_with_fallback: gdown (Drive) primary -> HTTP mirror, cached, LOUD ---
import os

def fetch_with_fallback(dest_name, gdrive_id, http_url, label):
    """Return a local cached path. If a real Google-Drive id is configured, try
    gdown first; otherwise (or on failure) download from the HTTP mirror — which
    is the GitHub Release asset. Caches to CACHE_DIR; prints which path was taken."""
    dest = os.path.join(CACHE_DIR, dest_name)
    if os.path.exists(dest):
        print(f"\u2713 {label}: using cached {dest}")
        return dest
    # Primary: gdown from course Drive — skipped when no real id is set (e.g. the
    # <GDRIVE_ID_*> placeholder), so GitHub-Release hosting is a clean primary.
    _has_gid = bool(gdrive_id) and not str(gdrive_id).startswith("<")
    if _has_gid:
        try:
            import gdown
            gdown.download(id=gdrive_id, output=dest, quiet=True)
            if os.path.exists(dest) and os.path.getsize(dest) > 0:
                print(f"\u2713 {label}: downloaded via gdown (Drive) -> {dest}")
                return dest
            raise RuntimeError("gdown produced no file")
        except Exception as exc:                          # noqa: BLE001
            print(f"\u26a0 {label}: gdown path failed ({exc}); FALLING BACK to HTTP mirror")
    # HTTP mirror = the GitHub Release asset (the primary path for this course).
    try:
        import urllib.request
        urllib.request.urlretrieve(http_url, dest)
        print(f"\u2713 {label}: downloaded via HTTP (release) -> {dest}")
        return dest
    except Exception as exc:                              # noqa: BLE001
        raise RuntimeError(
            f"{label}: fetch failed ({exc}). Is the asset uploaded to the "
            f"release? See KNOWN_ISSUES.md."
        ) from None

In [ ]:
# --- Masked metrics (mask zeros = missing, exactly as Unit 3) -----------------
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def masked_metrics(y_true, y_pred):
    """MAE / RMSE / MAPE with zero-speed entries masked out (METR-LA encodes
    missing as 0; MAPE explodes near zero). y_* are 1-D arrays in mph."""
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    m = y_true > 1e-6                      # mask zeros-as-missing
    yt, yp = y_true[m], y_pred[m]
    mae = mean_absolute_error(yt, yp)
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    mape = float(np.mean(np.abs((yt - yp) / yt)) * 100.0)
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape}

In [ ]:
# --- plot_error_distributions: box + histogram + ECDF (all three) ------------
import matplotlib.pyplot as plt

def plot_error_distributions(errors_by_model, horizon_label, units="mph"):
    """errors_by_model: dict name -> 1-D array of per-sample |error|.
    Draws box + overlaid histogram + ECDF side by side (show-don't-print)."""
    names = list(errors_by_model)
    data = [np.asarray(errors_by_model[n], float).ravel() for n in names]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    try:
        axes[0].boxplot(data, tick_labels=names, showfliers=False)  # mpl >= 3.9
    except TypeError:
        axes[0].boxplot(data, labels=names, showfliers=False)       # mpl < 3.9
    axes[0].set(title=f"Abs error spread @ {horizon_label}",
                ylabel=f"|error| ({units})")

    for n, d in zip(names, data):
        axes[1].hist(d, bins=40, alpha=0.5, label=n, density=True)
    axes[1].set(title="Error histogram", xlabel=f"|error| ({units})",
                ylabel="density")
    axes[1].legend()

    for n, d in zip(names, data):
        ds = np.sort(d)
        ecdf = np.arange(1, len(ds) + 1) / len(ds)
        axes[2].plot(ds, ecdf, label=n)
    axes[2].set(title="Error ECDF (more left/up = more small errors)",
                xlabel=f"|error| ({units})", ylabel="cumulative fraction")
    axes[2].legend()
    fig.tight_layout()
    return fig

def plot_loss_curve(losses, title="Training loss"):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(range(1, len(losses) + 1), losses, marker="o")
    ax.set(title=title, xlabel="epoch", ylabel="MSE loss")
    fig.tight_layout()
    return fig

## Beat 1 — The shockwave and the spatial gap Unit 3 measured

We load **METR-LA** as a spatio-temporal graph: the *graph is fixed* (207 sensor
nodes, road-network adjacency) and the *node features stream over time* (speed +
a normalized time-of-day channel, every 5 min). Then we find a real congestion
**shockwave** — a speed drop that appears at an **upstream** sensor and reaches a
**downstream** sensor a few minutes later. That lag is the spatial signal a
univariate model can't see, and the budget this GNN must earn.


In [ ]:
# --- Load METR-LA via the bundled loader (auto-downloads; LOUD fallback) -----
import torch
from torch_geometric_temporal.dataset import METRLADatasetLoader
from torch_geometric_temporal.signal import temporal_signal_split

NUM_IN, NUM_OUT = 12, 12          # 12 past steps (60 min) -> 12 future steps
try:
    loader = METRLADatasetLoader()                       # auto-downloads METR-LA
    dataset = loader.get_dataset(num_timesteps_in=NUM_IN, num_timesteps_out=NUM_OUT)
    print("✓ loaded METR-LA via METRLADatasetLoader (auto-download)")
except Exception as exc:                                 # noqa: BLE001
    print(f"⚠ METRLADatasetLoader failed ({exc}); FALLING BACK to cached mirror")
    # Hosted fallback: a small cached tensor bundle (same node/feature layout).
    _p = fetch_with_fallback("metrla_fallback.pt", "<GDRIVE_ID_metrla>",
                             "https://github.com/bgalon/geo-graph-learning/releases/download/u5-data/metrla_fallback.pt",
                             label="METR-LA fallback")
    raise RuntimeError(
        "Bundled loader unavailable — load the cached bundle per KNOWN_ISSUES.md."
    ) from None

train_dataset, test_dataset = temporal_signal_split(dataset, train_ratio=0.8)

# One snapshot tells us the graph shape.
_snap = next(iter(dataset))
edge_index = _snap.edge_index
NUM_NODES = _snap.x.shape[0]
NUM_FEATURES = _snap.x.shape[1]
print(f"  graph: {NUM_NODES} nodes, edge_index {tuple(edge_index.shape)}, "
      f"node features per step: {NUM_FEATURES} (speed + time-of-day)")
print("  5-min spacing, Mar–Jun 2012. NOTE: zeros encode MISSING speed — "
      "we mask them before any metric (same gotcha as Unit 3).")

In [ ]:
# --- Speed matrix (real mph) + ALL sensor coordinates -------------------------
import numpy as np, pandas as pd, os

# A REAL adjacent upstream/downstream pair (data-selected: strong road-adjacency
# in loader.A AND a positive-lag cross-correlation, so upstream genuinely leads).
# nodes 0/1 were arbitrary indices where upstream never congested - wrong story.
TARGET_NODE = 70       # downstream sensor (the one we forecast)
UPSTREAM_NODE = 176    # physically upstream (1-hop adjacent; leads by ~20 min)

# REAL mph: the loader downloads the raw speed matrix to disk (node_values.npy)
# BEFORE z-scoring it in memory, so read that to plot/map in mph. Fall back to the
# z-scored snapshots if the raw file isn't present.
speeds, SPEED_UNIT = None, "mph"
try:
    _nv = np.load(os.path.join(loader.raw_data_dir, "node_values.npy"))   # [T, nodes, 2] mph
    speeds = _nv[:, :, 0].T                                               # [nodes, T]
except Exception:
    speeds = np.stack([s.x[:, 0, 0].numpy() for s in dataset], axis=1)
    SPEED_UNIT = "normalized speed (z-score)"
print(f"  speed matrix: {speeds.shape} (nodes x timesteps); unit = {SPEED_UNIT}")

# Real clock time: METR-LA starts 2012-03-01 00:00, one sample every 5 min.
TIMESTAMPS = pd.date_range("2012-03-01", periods=speeds.shape[1], freq="5min")

# Sensor lat/lon for ALL nodes (DCRNN graph_sensor_locations.csv; row order = node order).
_COORDS_URL = ("https://raw.githubusercontent.com/liyaguang/DCRNN/master/"
               "data/sensor_graph/graph_sensor_locations.csv")
try:
    _loc = pd.read_csv(_COORDS_URL)
    SENSOR_LATLON = {i: (float(r["latitude"]), float(r["longitude"]))
                     for i, (_, r) in enumerate(_loc.iterrows())}
    if len(SENSOR_LATLON) < NUM_NODES:
        raise ValueError(f"only {len(SENSOR_LATLON)} coords for {NUM_NODES} nodes")
    print(f"\u2713 loaded {len(SENSOR_LATLON)} METR-LA sensor coordinates")
except Exception as exc:                            # noqa: BLE001
    print(f"\u26a0 sensor-location fetch failed ({exc}); FALLING BACK to a synthetic layout")
    _rng = np.random.default_rng(0)
    SENSOR_LATLON = {i: (34.05 + 0.15 * _rng.random(), -118.30 + 0.15 * _rng.random())
                     for i in range(NUM_NODES)}
print(f"  target node {TARGET_NODE} (downstream), upstream node {UPSTREAM_NODE}")

# z-score scaler the LOADER used (recomputed from the raw speed channel) so we can
# convert the models' NORMALIZED outputs back to mph.  mph = z*STD + MEAN
try:
    SPEED_MEAN = float(_nv[:, :, 0].mean()); SPEED_STD = float(_nv[:, :, 0].std())
except Exception:
    SPEED_MEAN, SPEED_STD = 53.72, 20.26      # METR-LA canonical
def to_mph(z):   return np.asarray(z, float) * SPEED_STD + SPEED_MEAN
def to_norm(mph): return (np.asarray(mph, float) - SPEED_MEAN) / SPEED_STD
print(f"  z-score scaler: mean={SPEED_MEAN:.1f} mph, std={SPEED_STD:.1f} mph")

### A note on units — real mph vs *normalized* speed

The plots are in **mph**, but the GNN does **not** train on mph. Neural nets learn
best when every input is centered on a similar scale, so the loader
**z-score-normalizes** each speed reading:

$$ z = \frac{x - \mu}{\sigma}, \qquad \mu \approx 53.7\ \text{mph},\quad \sigma \approx 20.3\ \text{mph} $$

so free-flow (~65 mph) becomes \(z \approx +0.6\) and a jam (~10 mph) becomes
\(z \approx -2.1\). Two consequences we handle explicitly below:
1. The model **predicts in z-score space** — we **de-normalize** back to mph
   (\(x = z\,\sigma + \mu\)) before plotting or scoring, so results are readable
   and comparable to Unit 3's **mph** SARIMA/HA baselines.
2. "Missing" readings are stored as **0 mph** (\(z \approx -2.65\)); we mask them
   out of every metric.

In [ ]:
# --- Normalization at a glance: raw mph vs z-scored distribution --------------
import matplotlib.pyplot as plt
_valid = speeds[speeds > 0]                        # ignore missing (stored as 0)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.5))
a1.hist(_valid.ravel(), bins=60, color="#0072b2")
a1.axvline(SPEED_MEAN, color="k", ls="--", label=f"mean {SPEED_MEAN:.0f} mph")
a1.set(title="Raw speed (mph)", xlabel="mph", ylabel="count"); a1.legend()
a2.hist(to_norm(_valid).ravel(), bins=60, color="#e0218a")
a2.axvline(0, color="k", ls="--", label="mean = 0")
a2.set(title="After z-score normalization", xlabel="z = (mph \u2212 mean) / std"); a2.legend()
fig.suptitle("Same data, two scales \u2014 the GNN trains on the right one")
fig.tight_layout()

*What to notice:* the shock appears **upstream first**, then shows up at the
downstream sensor a few minutes later — time is moving **through space**. That
upstream→downstream lag is exactly the product a univariate SARIMA (Unit 3)
cannot model, and what the graph model is being asked to recover.


In [ ]:
# --- V2a(i): the shockwave \u2014 real mph (left) + normalized z-score (right) ----
import matplotlib.pyplot as plt
def _mask0(a): return np.where(np.asarray(a, float) > 0, a, np.nan)

ds_speed = _mask0(speeds[TARGET_NODE])
split = int(0.8 * speeds.shape[1]); win = 60
drop_idx = split + int(np.nanargmin(np.diff(ds_speed[split:split + 2000])))
lo = max(0, drop_idx - win); hi = min(speeds.shape[1], drop_idx + win)
SHOCK = (lo, hi, drop_idx)
t = TIMESTAMPS[lo:hi]; WEEK = 7 * 24 * 12
up, dn = _mask0(speeds[UPSTREAM_NODE, lo:hi]), _mask0(speeds[TARGET_NODE, lo:hi])

fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 4))
# LEFT: real mph (+ same window one week earlier)
axL.plot(t, up, color="#e0218a", lw=2, label=f"upstream ({UPSTREAM_NODE})")
axL.plot(t, dn, color="#0072b2", lw=2, label=f"downstream ({TARGET_NODE})")
if lo - WEEK >= 0:
    axL.plot(t, _mask0(speeds[UPSTREAM_NODE, lo-WEEK:hi-WEEK]), color="#e0218a", ls=":", lw=1.3, alpha=.6, label="upstream, prev week")
    axL.plot(t, _mask0(speeds[TARGET_NODE, lo-WEEK:hi-WEEK]), color="#0072b2", ls=":", lw=1.3, alpha=.6, label="downstream, prev week")
axL.axvline(t[drop_idx - lo], color="k", ls="--", lw=1)
axL.set(title="Real speed (mph)", xlabel="time", ylabel="speed (mph)")
axL.legend(fontsize=7, loc="lower left")
# RIGHT: normalized z-score (exactly what the GNN sees)
axR.plot(t, to_norm(up), color="#e0218a", lw=2)
axR.plot(t, to_norm(dn), color="#0072b2", lw=2)
axR.axhline(0, color="#999", lw=.8); axR.axvline(t[drop_idx - lo], color="k", ls="--", lw=1)
axR.set(title="Normalized z-score (what the GNN trains on)", xlabel="time",
        ylabel="z = (mph \u2212 mean) / std")
fig.suptitle("Congestion shockwave: upstream leads downstream \u2014 and it's not the usual weekly pattern")
fig.autofmt_xdate(); fig.tight_layout()

*What to notice:* "upstream" is a **real place on the road**, not an
abstraction. Toggle the basemaps (incl. the blank **topology-only** layer) to see
the sensors with and without map context. Marker color encodes current speed
(red = slow); size marks the target vs. upstream.


In [ ]:
# --- V2a(ii): interactive folium map of the two sensors (labelled) ------------
import folium

pair = {TARGET_NODE: SENSOR_LATLON[TARGET_NODE],
        UPSTREAM_NODE: SENSOR_LATLON[UPSTREAM_NODE]}
ctr_lat = np.mean([v[0] for v in pair.values()])
ctr_lon = np.mean([v[1] for v in pair.values()])

m = folium.Map(location=[ctr_lat, ctr_lon], zoom_start=13, tiles=None)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer("CartoDB positron", name="Carto light").add_to(m)
folium.TileLayer(tiles="", attr="topology-only", name="Blank (topology only)").add_to(m)

now = SHOCK[2]                                       # shock-onset timestep (in range)
_lo_ref, _hi_ref = np.nanpercentile(speeds[:, now], [5, 95])
def speed_color(v):                                 # red = slow, green = fast
    f = 0.0 if _hi_ref == _lo_ref else max(0.0, min(1.0, (v - _lo_ref) / (_hi_ref - _lo_ref)))
    return f"#{int(255*(1-f)):02x}{int(255*f):02x}33"

for node, (la, lo_) in pair.items():
    role = "downstream (TARGET)" if node == TARGET_NODE else "upstream"
    spd = float(speeds[node, now])
    folium.CircleMarker(
        location=[la, lo_],
        radius=12 if node == TARGET_NODE else 8,
        color="black", weight=2, fill=True,
        fill_color=speed_color(spd), fill_opacity=0.9,
        popup=f"node {node} \u2014 {spd:.0f} {SPEED_UNIT} ({role})",
    ).add_to(m)
    # ALWAYS-VISIBLE label so you can see which node is which without clicking
    folium.map.Marker(
        [la, lo_],
        icon=folium.DivIcon(html=(
            f'<div style="font-size:12px;font-weight:700;color:#111;'
            f'background:rgba(255,255,255,.75);padding:1px 4px;border-radius:3px;'
            f'transform:translate(12px,-8px);white-space:nowrap">'
            f'node {node} \u00b7 {role}</div>')),
    ).add_to(m)
folium.PolyLine([pair[UPSTREAM_NODE], pair[TARGET_NODE]],
                color="crimson", weight=3, opacity=0.8,
                tooltip="shock propagates upstream \u2192 downstream").add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m

In [ ]:
# --- V2b: the whole METR-LA sensor NETWORK as a graph over a map --------------
# All nodes + road-adjacency EDGES (the substrate messages pass along), colored
# by speed at the shock onset. This is the graph the GNN actually operates on.
import folium

now = SHOCK[2]
sp_now = speeds[:, now]                              # per-node speed at onset
_lo, _hi = np.nanpercentile(sp_now, [5, 95])
def _col(v):                                         # red = slow, green = fast
    f = 0.0 if _hi == _lo else max(0.0, min(1.0, (v - _lo) / (_hi - _lo)))
    return f"#{int(255*(1-f)):02x}{int(255*f):02x}33"

_ctr = [np.mean([c[0] for c in SENSOR_LATLON.values()]),
        np.mean([c[1] for c in SENSOR_LATLON.values()])]
gm = folium.Map(location=_ctr, zoom_start=11, tiles=None)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(gm)
folium.TileLayer("CartoDB positron", name="Carto light").add_to(gm)
folium.TileLayer(tiles="", attr="topology-only", name="Blank (topology only)").add_to(gm)

# edges: the road adjacency the GNN passes messages along (dedup undirected pairs)
_edges = folium.FeatureGroup(name="graph edges (road adjacency)").add_to(gm)
_ei = edge_index.detach().cpu().numpy()
_seen = set()
for a, b in zip(_ei[0].tolist(), _ei[1].tolist()):
    if a == b or (b, a) in _seen or a not in SENSOR_LATLON or b not in SENSOR_LATLON:
        continue
    _seen.add((a, b))
    folium.PolyLine([SENSOR_LATLON[a], SENSOR_LATLON[b]],
                    color="#999", weight=1, opacity=0.4).add_to(_edges)

# nodes: colored by speed at the shock onset; target/upstream ringed + enlarged
_nodes = folium.FeatureGroup(name=f"sensors (speed @ onset, {SPEED_UNIT})").add_to(gm)
for i, (la, lo_) in SENSOR_LATLON.items():
    if i >= speeds.shape[0]:
        continue
    hl = i in (TARGET_NODE, UPSTREAM_NODE)
    tag = (" — TARGET/downstream" if i == TARGET_NODE
           else " — UPSTREAM" if i == UPSTREAM_NODE else "")
    folium.CircleMarker(
        location=[la, lo_],
        radius=8 if hl else 3,
        color="black" if hl else "#00000022", weight=2 if hl else 0,
        fill=True, fill_color=_col(float(sp_now[i])), fill_opacity=0.9,
        popup=f"node {i}: {sp_now[i]:.1f} {SPEED_UNIT}{tag}",
    ).add_to(_nodes)
folium.LayerControl(collapsed=False).add_to(gm)
print(f"network map: {len(_seen)} undirected edges over "
      f"{sum(1 for i in SENSOR_LATLON if i < speeds.shape[0])} nodes "
      f"(toggle basemaps + the edges/sensors layers, top-right)")
gm

## Beat 2 — Message passing: GCN through GAT

**One abstraction, two instances.** *Message passing*: each node gathers a
**message** from its neighbors, then **updates** its own state. A **GCN** layer is
the simplest instance — the degree-normalized **average** of a node's neighbors
("smear each segment's speed toward its road neighbors"). **GAT** swaps the fixed
averaging for **learned, per-edge attention** (the prototype for interrogable
attention). Crucially: **k layers = a k-hop receptive field** — after k rounds,
each node has heard from everything k hops away.


*What to notice:* a single "hot" node's value **diffuses** outward — 1 hop
after one GCN layer, 2 hops after two. That spreading IS the spatial mechanism a
GRU-alone (time only) lacks. No training here — this is pure mechanism.


In [ ]:
# --- V3a: message passing = smearing (one GCNConv, run 1-hop then 2-hop) ------
import torch
from torch_geometric.nn import GCNConv

# A toy 1-channel signal: one node is "hot" (1.0), all others 0.
x0 = torch.zeros(NUM_NODES, 1)
x0[UPSTREAM_NODE] = 1.0

# A fixed (untrained) GCNConv just to visualize neighbor-averaging structure.
conv = GCNConv(1, 1, bias=False)
with torch.no_grad():
    conv.lin.weight.fill_(1.0)            # identity-ish: pure neighbor averaging
    h1 = conv(x0, edge_index)             # after 1 hop
    h2 = conv(h1, edge_index)             # after 2 hops

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
for ax, (vec, ttl) in zip(axes, [(x0, "start (1 hot node)"),
                                 (h1, "after 1 GCN layer (1-hop)"),
                                 (h2, "after 2 GCN layers (2-hop)")]):
    v = vec.detach().numpy().ravel()
    nz = np.argsort(v)[-15:]              # show the most-activated nodes
    ax.bar(range(len(nz)), v[nz])
    ax.set(title=ttl, xlabel="node (top-activated)", ylabel="signal")
fig.suptitle("k rounds of message passing = a k-hop receptive field")
fig.tight_layout()

**GAT (named, not trained here).** GAT computes a *learned weight per edge*
— the first place the model says "this neighbor matters more than that one." That
is the prototype for the interrogable-attention idea we meet in A3T-GCN. (A
**spatial** GAT layer is one honest way to answer "which upstream node?"; the demo
takes a different, cleaner route below — see Beat 4.)


In [ ]:
# --- Build the T-GCN -----------------------------------------------------------
import torch
import torch.nn.functional as F
from torch_geometric_temporal.nn.recurrent import TGCN

USE_PRETRAINED = True         # default: load checkpoint (class). False = live-train moment.
EPOCHS = 6                    # 5-8 fits the ~10-12 min T4 budget
HIDDEN = 32

class TGCNForecaster(torch.nn.Module):
    """T-GCN encoder + linear head mapping hidden state -> NUM_OUT horizon.

    Each METR-LA snapshot x is [nodes, features, periods]; the GRU consumes the
    periods ONE STEP AT A TIME, carrying hidden state H (space, then time). A 2-D
    [nodes, features] input (e.g. the bus teaser) is treated as a single step."""
    def __init__(self, node_features, hidden, horizon):
        super().__init__()
        self.recurrent = TGCN(in_channels=node_features, out_channels=hidden)
        self.head = torch.nn.Linear(hidden, horizon)

    def forward(self, x, edge_index, edge_weight=None):
        if x.dim() == 2:                      # [nodes, features] -> one period
            x = x.unsqueeze(-1)
        H = None
        for step in range(x.shape[-1]):       # step the GRU across the window
            H = self.recurrent(x[:, :, step], edge_index, edge_weight, H)
        return self.head(F.relu(H))

device = torch.device(_DEVICE)
model = TGCNForecaster(NUM_FEATURES, HIDDEN, NUM_OUT).to(device)
edge_index = edge_index.to(device)

# --- WEIGHTED edges: use the road graph's real gaussian-kernel weights ---------
# The per-snapshot `edge_attr` is malformed for GCNConv's gcn_norm on some
# PyG-Temporal builds (raises IndexError). So build a validated [num_edges]
# weight vector ONCE: prefer the static dataset.edge_weight; else reconstruct
# from the adjacency loader.A; else fall back to an unweighted graph (None).
def _build_edge_weight():
    ew = getattr(dataset, "edge_weight", None)
    if ew is not None:
        ew = torch.as_tensor(np.asarray(ew), dtype=torch.float).view(-1)
        if ew.numel() == edge_index.shape[1]:
            return ew, "dataset.edge_weight"
    A = getattr(loader, "A", None)
    if A is not None:
        A = np.asarray(A); _ei = edge_index.cpu().numpy()
        return torch.tensor(A[_ei[0], _ei[1]], dtype=torch.float), "reconstructed from loader.A"
    return None, "unweighted (no valid weights found)"

EDGE_WEIGHT, _eww = _build_edge_weight()
if EDGE_WEIGHT is not None:
    EDGE_WEIGHT = EDGE_WEIGHT.to(device)
print(f"T-GCN built: in={NUM_FEATURES}, hidden={HIDDEN}, horizon={NUM_OUT}, "
      f"device={_DEVICE}")
print(f"  edges: {'WEIGHTED via ' + _eww if EDGE_WEIGHT is not None else 'unweighted (fallback)'}"
      + (f' [{EDGE_WEIGHT.numel()} weights]' if EDGE_WEIGHT is not None else ''))

In [ ]:
# --- Live-train OR load pre-trained (USE_PRETRAINED switch) --------------------
import time

# METR-LA yields ~27k HEAVILY OVERLAPPING windows (stride 1). Training/evaluating
# on all of them is ~30-45 min/epoch — far over budget. Subsample every STRIDE-th
# window (still spans the whole period). Set *_STRIDE=1 for a full run.
TRAIN_STRIDE = 12
TEST_STRIDE = 6
train_windows = list(train_dataset)[::TRAIN_STRIDE]
test_windows = list(test_dataset)[::TEST_STRIDE]
print(f"eval subset: {len(test_windows)} windows (every {TEST_STRIDE}th)")

losses = []
_loaded = False
if USE_PRETRAINED:
    try:
        path = fetch_with_fallback("tgcn_metrla.pt", CKPT["tgcn_metrla"]["gdrive_id"],
                                   CKPT["tgcn_metrla"]["http"], label="T-GCN checkpoint")
        model.load_state_dict(torch.load(path, map_location=device))
        _loaded = True
        print("\u2713 USING PRE-TRAINED tgcn_metrla.pt (skipped live-train). "
              "Set USE_PRETRAINED=False for the in-class live-train moment.")
    except Exception as exc:                              # noqa: BLE001
        print(f"\u26a0 pretrained T-GCN unavailable ({exc}); FALLING BACK to live-train")
if not _loaded:
    print(f"live-training T-GCN on {len(train_windows)} windows "
          f"(every {TRAIN_STRIDE}th) x {EPOCHS} epochs...")
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    t0 = time.time()
    for epoch in range(EPOCHS):
        model.train(); ep_loss = 0.0; n = 0
        for snap in train_windows:
            x = snap.x.to(device)
            y = snap.y.to(device)                  # [nodes, horizon]
            opt.zero_grad()
            yhat = model(x, edge_index, EDGE_WEIGHT)
            loss = F.mse_loss(yhat, y)
            loss.backward(); opt.step()
            ep_loss += loss.item(); n += 1
        losses.append(ep_loss / max(n, 1))
        print(f"  epoch {epoch+1}/{EPOCHS}  loss={losses[-1]:.4f}  "
              f"({time.time()-t0:.0f}s elapsed)")
    print("\u2713 LIVE-TRAINED T-GCN")

*What to notice:* the loss **falling** is the GCN-inside-GRU learning
spatial **and** temporal structure at once — something neither a GCN alone (no
time) nor a GRU alone (no space) can do. (Empty if you used the checkpoint.)


In [ ]:
# --- V4a: the live loss curve --------------------------------------------------
if losses:
    plot_loss_curve(losses, title="T-GCN live training loss (MSE)")
else:
    print("Used pre-trained checkpoint — no live loss curve. "
          "Set USE_PRETRAINED=False and re-run to train live.")

*What to notice:* did the T-GCN **see the shock coming**? Its 15-min-ahead
forecast (3 steps) over the shockwave window, overlaid on the actual downstream
speed — the qualitative read before the quantitative eval.


In [ ]:
# --- V4b: predicted-vs-actual on the shockwave window (15-min horizon) --------
# The model predicts in z-score space; we show BOTH the normalized output (its
# native units) and the de-normalized mph (what the number actually means).
model.eval()
H15 = 2          # 15 min ahead = step index 2 (3 steps @ 5 min)
preds, actuals = [], []
with torch.no_grad():
    for snap in test_windows:
        x = snap.x.to(device)
        yhat = model(x, edge_index, EDGE_WEIGHT)
        preds.append(float(yhat[TARGET_NODE, H15].cpu()))
        actuals.append(float(snap.y[TARGET_NODE, H15]))

import matplotlib.pyplot as plt
preds_z, actuals_z = np.array(preds), np.array(actuals)     # z-score (native)
preds_mph, actuals_mph = to_mph(preds_z), to_mph(actuals_z) # de-normalized to mph
s0 = max(0, len(preds_z)//2 - 80); s1 = s0 + 160

fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 4))
axL.plot(actuals_mph[s0:s1], label="actual"); axL.plot(preds_mph[s0:s1], label="T-GCN forecast")
axL.set(title="Real speed (mph)", xlabel="test timestep (5-min)", ylabel="speed (mph)"); axL.legend()
axR.plot(actuals_z[s0:s1], label="actual"); axR.plot(preds_z[s0:s1], label="T-GCN forecast")
axR.axhline(0, color="#999", lw=.8)
axR.set(title="Normalized (model's native output)", xlabel="test timestep (5-min)", ylabel="z-score"); axR.legend()
fig.suptitle("Did the T-GCN see the shock coming? (15-min-ahead forecast)")
fig.tight_layout()

## Beat 3 — T-GCN = GCN + GRU

T-GCN puts a **GCN inside a GRU**: at each step the GCN produces spatially-aware
node features; the GRU carries state across steps. Two receptive-field axes now:
**space** (GCN hops) x **time** (GRU steps). The shockwave lives in both.

**`USE_PRETRAINED` switch (default `True`).** For class we **load** the
pre-trained `tgcn_metrla.pt` from the course release - instant and reliable. To
run the live "watch the loss fall" moment instead, set **`USE_PRETRAINED = False`**:
it live-trains on a stride-subsampled subset (`TRAIN_STRIDE`, ~10-15 min on a free
T4) and falls back to the checkpoint if the GPU is slow. **A3T-GCN is *always*
loaded** (attention ~doubles training time), never live-trained.

In [ ]:
# --- Per-horizon errors (mph): T-GCN + a real PERSISTENCE baseline ------------
# Model predicts z-scores -> de-normalize to mph, mask missing (raw 0 mph).
# PERSISTENCE ("naive"): predict the speed STAYS at its current value. A real,
# computable baseline that DEGRADES with horizon (unlike the flat HA floor) \u2014
# great at 15 min, poor at 60. "now" = the last input step of the window.
HORIZON_STEPS = {"15 min": 2, "30 min": 5, "60 min": 11}
tgcn_err = {}; tgcn_metrics = {}; persist_err = {}; persist_metrics = {}
model.eval()
with torch.no_grad():
    buf = {h: ([], []) for h in HORIZON_STEPS}; now_buf = []
    for snap in test_windows:
        x = snap.x.to(device); yhat = model(x, edge_index, EDGE_WEIGHT)
        now_buf.append(float(snap.x[TARGET_NODE, 0, -1]))     # current speed (z), last input step
        for h, step in HORIZON_STEPS.items():
            buf[h][0].append(float(yhat[TARGET_NODE, step].cpu()))
            buf[h][1].append(float(snap.y[TARGET_NODE, step]))
now_z = np.array(now_buf)
for h, (pz, az) in buf.items():
    p_mph, a_mph = to_mph(np.array(pz)), to_mph(np.array(az))
    per_mph = to_mph(now_z)                                    # persistence prediction (mph)
    m = a_mph > 1.0
    tgcn_metrics[h]    = masked_metrics(a_mph[m], p_mph[m]);   tgcn_err[h]    = np.abs(a_mph[m] - p_mph[m])
    persist_metrics[h] = masked_metrics(a_mph[m], per_mph[m]); persist_err[h] = np.abs(a_mph[m] - per_mph[m])
print("Per-horizon RMSE (mph):   T-GCN / persistence")
for h in HORIZON_STEPS:
    print(f"  {h}: {tgcn_metrics[h]['RMSE']:.2f} / {persist_metrics[h]['RMSE']:.2f}")

In [ ]:
# --- Baselines: REAL HA floor + REAL persistence + OPTIONAL SARIMA ------------
# HA = seasonal mean per 5-min-of-week (flat floor, learned on train). Persistence
# was computed above (degrades with horizon). BOTH ARE REAL. SARIMA is OPTIONAL \u2014
# only your Unit-3 run produces it; leave it None to skip, or paste your dict.
WEEK = 7 * 24 * 12
_sp = speeds[TARGET_NODE]; _slot = np.arange(_sp.shape[0]) % WEEK
_split = int(0.8 * _sp.shape[0]); _tr = np.arange(_sp.shape[0]) < _split
_ha_table = np.full(WEEK, np.nan)
for s in range(WEEK):
    m = (_slot == s) & (_sp > 0) & _tr
    if m.any(): _ha_table[s] = _sp[m].mean()
_ha_table[np.isnan(_ha_table)] = _sp[(_sp > 0) & _tr].mean()
_te = np.arange(_split, _sp.shape[0]); _m = _sp[_te] > 0
_ha_pred, _ha_act = _ha_table[_slot[_te]][_m], _sp[_te][_m]
HA_RMSE = float(np.sqrt(np.mean((_ha_act - _ha_pred) ** 2))); ha_err_all = np.abs(_ha_act - _ha_pred)
U3_HA_RMSE  = {h: HA_RMSE for h in HORIZON_STEPS}
PERSIST_RMSE = {h: persist_metrics[h]["RMSE"] for h in HORIZON_STEPS}
TGCN_RMSE    = {h: tgcn_metrics[h]["RMSE"] for h in HORIZON_STEPS}

# OPTIONAL: paste your Unit-3 SARIMA per-horizon RMSE to overlay it; else None.
U3_SARIMA_RMSE = None      # e.g. {"15 min": 5.6, "30 min": 7.1, "60 min": 8.4}

print(f"REAL floors (mph):  HA {HA_RMSE:.2f} (flat)")
print("  horizon:  T-GCN / persistence / HA" + ("  / SARIMA" if U3_SARIMA_RMSE else ""))
for h in HORIZON_STEPS:
    line = f"  {h}: {TGCN_RMSE[h]:.2f} / {PERSIST_RMSE[h]:.2f} / {HA_RMSE:.2f}"
    if U3_SARIMA_RMSE: line += f" / {U3_SARIMA_RMSE[h]:.2f}"
    print(line)

*What to notice:* where the T-GCN curve sits relative to SARIMA and the flat
HA floor **is** the "did the GNN earn its complexity" answer. If the GNN's win is
largest at 15 min and shrinks by 60 min, that's the honest, on-thesis result —
the neighbor signal lives at short horizons.


### The baselines the GNN must beat \u2014 and why "SARIMA \u2248 HA" is the point

Two **real, computed** floors bracket the univariate space:
- **Historical average (HA)** \u2014 predict each sensor's mean speed for this
  5-min-of-week slot. **Horizon-independent** (a *flat* floor) and, on METR-LA,
  surprisingly **strong**, because traffic is dominated by its daily/weekly cycle.
- **Persistence** \u2014 predict "speed stays what it is now." **Degrades with horizon**:
  great at 15 min, poor at 60. The crossover between persistence and the flat HA
  floor *is* the classic short-vs-long-horizon tradeoff.

A GNN "earns its complexity" only if it beats **both** \u2014 and the honest result on
METR-LA is that these dumb floors are hard to beat, because a single sensor's
*own* history already captures most of the daily pattern. The GNN's edge has to
come from the **spatial** signal (the upstream neighbor) that the floors are blind
to \u2014 largest exactly at the horizons where that neighbor's congestion arrives.

**SARIMA is optional here.** It's your Unit-3 *smart univariate* model, kept only
for continuity ("does the GNN beat what *you* built?"). It is fragile/expensive on
5-min data and rarely beats the HA floor by much \u2014 which is itself the breakeven
lesson. Set `U3_SARIMA_RMSE` to your real U3 numbers to overlay it, or leave it off.

In [ ]:
# --- V5a: breakeven \u2014 does the GNN beat the REAL dumb floors? ------------------
import matplotlib.pyplot as plt
hs = list(HORIZON_STEPS); xs = [15, 30, 60]
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(xs, [TGCN_RMSE[h] for h in hs], "o-", label="T-GCN (graph)")
ax.plot(xs, [PERSIST_RMSE[h] for h in hs], "d-", color="#888", label="persistence (naive, degrades)")
ax.plot(xs, [U3_HA_RMSE[h] for h in hs], ":", color="#333", lw=2, label="historical-average floor (flat)")
if U3_SARIMA_RMSE:
    ax.plot(xs, [U3_SARIMA_RMSE[h] for h in hs], "s--", label="your U3 SARIMA")
ax.set(title="Breakeven: does the GNN beat the dumb floors?",
       xlabel="forecast horizon (min)", ylabel="RMSE (mph)")
ax.text(0.5, 1.06, "HA + persistence are REAL (computed here). T-GCN reflects YOUR checkpoint; "
        "SARIMA is optional (your U3 number).", transform=ax.transAxes, ha="center", va="bottom",
        fontsize=7.5, color="#555")
ax.legend(loc="upper left"); fig.tight_layout()

*What to notice:* the comparison is **three spreads**, not three means. The
boxplot shows medians/outliers, the histogram the shape, the ECDF "which model
has more small errors." If means crossed but the ECDF disagrees, trust the ECDF.


In [ ]:
# --- V5b: error distribution at 30 min (box + histogram + ECDF) ---------------
# All three are REAL per-sample |error| arrays (T-GCN, persistence, HA). SARIMA is
# added only if you provided U3 numbers (then synthesized from its RMSE).
err_30 = {"T-GCN": tgcn_err["30 min"], "persistence": persist_err["30 min"], "HA": ha_err_all}
if U3_SARIMA_RMSE:
    _rng = np.random.default_rng(0)
    err_30["SARIMA"] = np.abs(_rng.normal(0, U3_SARIMA_RMSE["30 min"], len(tgcn_err["30 min"])))
_f = plot_error_distributions(err_30, horizon_label="30 min")
_f.text(0.5, 1.02, "T-GCN, persistence, HA are REAL per-sample errors "
        "(SARIMA, if shown, is synthesized from your placeholder RMSE).",
        ha="center", fontsize=8, color="#555")

## Beat 4 — A3T-GCN: attention on top of T-GCN

A3T-GCN adds an **attention layer** over the time dimension: it learns a weight
for **each past time step** and assembles a weighted history before forecasting.
We **load** the pre-trained model (`a3tgcn_metrla.pt`, built against the
non-batched **`A3TGCN`** class) — **never trained live** — and compare it to plain
T-GCN with the same protocol.

> **Say it before any heatmap:** A3T-GCN's attention is over **time steps**, not
> over neighbor **nodes**. We frame it correctly now and interrogate it honestly
> in the next section.


In [ ]:
# --- Load the pre-trained A3T-GCN (always loaded, never live-trained) ---------
from torch_geometric_temporal.nn.recurrent import A3TGCN

class A3TGCNForecaster(torch.nn.Module):
    def __init__(self, node_features, hidden, periods, horizon):
        super().__init__()
        self.recurrent = A3TGCN(in_channels=node_features,
                                out_channels=hidden, periods=periods)
        self.head = torch.nn.Linear(hidden, horizon)

    def forward(self, x_seq, edge_index, edge_weight=None):
        # x_seq: [nodes, features, periods]
        h = self.recurrent(x_seq, edge_index, edge_weight)
        return self.head(F.relu(h))

a3t = A3TGCNForecaster(NUM_FEATURES, HIDDEN, NUM_IN, NUM_OUT).to(device)
ck = fetch_with_fallback("a3tgcn_metrla.pt", CKPT["a3tgcn_metrla"]["gdrive_id"],
                         CKPT["a3tgcn_metrla"]["http"], label="A3T-GCN checkpoint")
a3t.load_state_dict(torch.load(ck, map_location=device))
a3t.eval()
print("✓ loaded A3T-GCN from a3tgcn_metrla.pt "
      "(checkpoint trained against the pinned PyG-T version — see KNOWN_ISSUES.md)")

*What to notice:* did attention buy accuracy? Overlay the A3T-GCN curve on
the breakeven plot. "Earned" is **measured**, not assumed — it may help only
modestly, and that's a legitimate result.


In [ ]:
# --- Evaluate A3T-GCN per horizon; V6a overlay on the breakeven plot ----------
a3t_metrics, a3t_err = {}, {}
buf2 = {h: ([], []) for h in HORIZON_STEPS}
with torch.no_grad():
    for snap in test_windows:
        x = snap.x.to(device)
        x_seq = x if x.dim() == 3 else x.unsqueeze(-1).repeat(1, 1, NUM_IN)
        yhat = a3t(x_seq, edge_index, EDGE_WEIGHT)
        for h, step in HORIZON_STEPS.items():
            buf2[h][0].append(float(yhat[TARGET_NODE, step].cpu()))
            buf2[h][1].append(float(snap.y[TARGET_NODE, step]))
for h, (pz, az) in buf2.items():
    p_mph, a_mph = to_mph(np.array(pz)), to_mph(np.array(az))   # de-normalize to mph
    m = a_mph > 1.0
    a3t_metrics[h] = masked_metrics(a_mph[m], p_mph[m]); a3t_err[h] = np.abs(a_mph[m] - p_mph[m])
A3T_RMSE = {h: a3t_metrics[h]["RMSE"] for h in HORIZON_STEPS}

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(xs, [TGCN_RMSE[h] for h in hs], "o-", label="T-GCN")
ax.plot(xs, [A3T_RMSE[h] for h in hs], "^-", label="A3T-GCN (attention)")
ax.plot(xs, [PERSIST_RMSE[h] for h in hs], "d-", color="#888", label="persistence")
ax.plot(xs, [U3_HA_RMSE[h] for h in hs], ":", color="#333", lw=2, label="HA floor (real)")
if U3_SARIMA_RMSE:
    ax.plot(xs, [U3_SARIMA_RMSE[h] for h in hs], "s--", label="your U3 SARIMA")
ax.set(title="Did attention earn its complexity? (A3T vs T-GCN fair ONLY if trained equally)",
       xlabel="forecast horizon (min)", ylabel="RMSE (mph)")
ax.text(0.5, 1.06, "A3T vs T-GCN reflects YOUR checkpoints \u2014 compare only if both trained the same.",
        transform=ax.transAxes, ha="center", va="bottom", fontsize=7.5, color="#555")
ax.legend(loc="upper left"); fig.tight_layout()

*What to notice:* A3T-GCN vs. T-GCN as **three spreads** at the horizon
where they differ most — same box/histogram/ECDF read as before.


In [ ]:
# --- V6b: A3T-GCN vs T-GCN error spread (box + histogram + ECDF) --------------
# pick the horizon with the biggest RMSE gap
gap_h = max(HORIZON_STEPS, key=lambda h: abs(TGCN_RMSE[h] - A3T_RMSE[h]))
plot_error_distributions(
    {"T-GCN": tgcn_err[gap_h], "A3T-GCN": a3t_err[gap_h]},
    horizon_label=gap_h)

## Beat 4 (cont.) — Attention on a real shockwave: temporal, NOT spatial

> ### The Face-(c) caveat (read before the visuals)
> A3T-GCN's attention is a **single global learned weighting over the 12 input
> time steps** — `softmax(A3TGCN._attention)`. It is **not per-sample** and it is
> **not over nodes**. So it can honestly tell you *"the model leans on the most
> recent ~k steps"* — a statement about **time**. It **cannot** tell you *"which
> upstream node"* — that is a **spatial** question. Presenting a temporal-
> attention heatmap as spatial causality is exactly the error this unit teaches
> against. The attention weight is an **observable hypothesis worth testing**, not
> a verified explanation.
>
> So we keep two panels on **separate axes**: the **temporal** attention bar, and
> Unit 3's **spatial** upstream→downstream cross-correlation. The "which node"
> story comes from the spatial panel, never from the attention bar.


In [ ]:
# --- Extract A3T-GCN's temporal attention (the global per-step weighting) ------
with torch.no_grad():
    attn = torch.nn.functional.softmax(a3t.recurrent._attention, dim=0)
    attn = attn.detach().cpu().numpy().ravel()        # shape [periods] = [12]
print(f"temporal attention weights (one per input step, sums to 1): "
      f"{np.round(attn, 3)}")

*What to notice (TIME):* the bars are weights over **past time steps** —
"how much the model leans on each of the last 12 readings." Bigger bars on recent
steps = a recency-weighted history. This says **nothing** about *which sensor*.


*What to notice (SPACE):* a peak at a **positive lag** means upstream speed
*now* predicts downstream speed *soon* — the **spatial** evidence for "which
upstream node." This is a different axis from the attention bar above; we never
merge them.


In [ ]:
# --- V7a: stacked panels — temporal attention (top) + U3 cross-corr (bottom) --
# Spatial diagnostic: lagged cross-correlation upstream -> downstream (Unit 3 idiom)
us = speeds[UPSTREAM_NODE]; dsx = speeds[TARGET_NODE]
maxlag = 12
ccf = []
for k in range(maxlag + 1):
    a = us[:len(us) - k]; b = dsx[k:]
    m = (a > 1e-6) & (b > 1e-6)
    ccf.append(np.corrcoef(a[m], b[m])[0, 1] if m.sum() > 2 else np.nan)
ccf = np.array(ccf)
peak_lag = int(np.nanargmax(ccf))

import matplotlib.pyplot as plt
fig, (axT, axS) = plt.subplots(2, 1, figsize=(9, 7))
axT.bar(range(1, len(attn) + 1), attn, color="#3b6")
axT.set(title="TEMPORAL — A3T-GCN attention over PAST TIME STEPS (global, not spatial)",
        xlabel="past input step (1=oldest ... 12=most recent)",
        ylabel="attention weight")
axS.plot(range(maxlag + 1), ccf, "o-", color="#36b")
axS.axvline(peak_lag, color="crimson", ls="--",
            label=f"peak at lag {peak_lag} (~{peak_lag*5} min)")
axS.set(title="SPATIAL — U3 upstream->downstream cross-correlation (the 'which node' axis)",
        xlabel="lag k (steps): upstream leads downstream by k",
        ylabel="correlation")
axS.legend(); fig.tight_layout()

## (OPTIONAL — trim if short on time) V7c — animated shockwave playback

The single biggest render cost in the notebook. Short clip (≈16 frames, low fps,
inline HTML5). It shows the **shock moving through space** (the map/series) while
the **attention bar stays fixed** — because the attention is a *global* temporal
prior, not a per-frame spatial signal. **Drop this cell first if the clock is
tight** (V7a above already makes the honest point).


*What to notice:* the shock **moves** frame to frame (space + time), but the
green attention bars **don't** — a visual proof that this attention is a fixed
temporal weighting, not a moving spatial spotlight.


In [ ]:
# --- V7c: animation (FuncAnimation -> to_jshtml; ~16 frames, low fps) ---------
import matplotlib.pyplot as plt
from matplotlib import animation

lo, hi, onset = SHOCK
frames = list(range(lo, min(hi, lo + 16)))         # cap frames for Colab render
fig, (axm, axa) = plt.subplots(1, 2, figsize=(11, 4))

def draw(frame):
    axm.clear(); axa.clear()
    axm.plot(speeds[UPSTREAM_NODE, lo:hi], color="#888", lw=1, label="upstream")
    axm.plot(speeds[TARGET_NODE, lo:hi], color="#333", lw=1, label="downstream")
    axm.axvline(frame - lo, color="crimson", lw=2)
    axm.set(title=f"shock moving through space+time (t={frame})",
            xlabel="step in window", ylabel="speed (mph)")
    axm.legend(loc="upper right")
    axa.bar(range(1, len(attn) + 1), attn, color="#3b6")
    axa.set(title="attention bar stays FIXED (global temporal prior)",
            xlabel="past step", ylabel="weight", ylim=(0, max(attn) * 1.2))

anim = animation.FuncAnimation(fig, draw, frames=frames, interval=200)
from IPython.display import HTML
plt.close(fig)
HTML(anim.to_jshtml())

## Beat 5 — When is a GNN overengineered? + the bus graph

A GNN earns its complexity when there's a **real, exploitable spatial signal** the
baseline can't see — and the win is **largest where the neighbor signal arrives**,
shrinking as the baseline strengthens (the Unit 3 caveat, now demonstrated in the
breakeven plot). The field itself admits diminishing returns from architectural
complexity (STAEformer, Liu et al. 2023,
[arXiv:2308.10425](https://arxiv.org/abs/2308.10425)).

A GNN is **overengineered** when the graph is tiny, the data thin, or the series
dominated by its own daily pattern — which is plausibly the case for a **~10-node
bus corridor over 17 days**. We re-frame Unit 3's corridor as a graph and show a
GNN *trains* on it. **That's all the demo does** — the real analysis is your
practice task.


*What to notice:* same road, **new object** — Unit 3's per-segment corridor
is now a small graph (nodes = segments, edges = adjacency). A tiny graph is
exactly where a GNN *might be overkill* — which makes it the perfect practice
substrate.


In [ ]:
# --- V8a: the bus corridor as a graph (fetch tiny graph; folium map) ----------
# BRIDGE from METR-LA (LA freeway SENSORS) to YOUR practice dataset: a Tel Aviv
# BUS corridor re-framed as a graph — nodes = road SEGMENTS, edges = adjacency,
# node features = per-segment speed. Same GNN machinery, new domain + your city.
import folium

try:
    bus_path = fetch_with_fallback("bus_corridor_u5.npz", BUS["gdrive_id"],
                                   BUS["http"], label="bus corridor graph")
    busz = np.load(bus_path, allow_pickle=True)
    bus_edge_index = torch.tensor(busz["edge_index"], dtype=torch.long)
    bus_speeds = busz["speeds"]                 # [segments, time]
    bus_latlon = np.asarray(busz["latlon"])     # [segments, 2]
    IS_REAL = True
    print(f"\u2713 bus graph: {bus_speeds.shape[0]} segment-nodes, "
          f"{bus_edge_index.shape[1]} edges (real corridor)")
except Exception as exc:                         # noqa: BLE001
    IS_REAL = False
    print(f"\u2139 bus graph not hosted yet \u2014 showing an ILLUSTRATIVE corridor along "
          f"Tel Aviv\u2019s Ibn Gabirol St; the real data-backed graph is a practice-task "
          f"artifact [{type(exc).__name__}]")
    # ~10 segment-nodes tracing a REAL Tel Aviv arterial (Ibn Gabirol) so it reads
    # as an actual bus corridor. Speeds are SYNTHETIC (a wave sweeping the line).
    bus_latlon = np.array([
        (32.07104, 34.78200), (32.07360, 34.78175), (32.07549, 34.78173),
        (32.07816, 34.78121), (32.07954, 34.78125), (32.08155, 34.78127),
        (32.08453, 34.78157), (32.08809, 34.78238), (32.09006, 34.78275),
        (32.09180, 34.78292)])
    N = len(bus_latlon); T = 300
    bus_edge_index = torch.tensor([list(range(N - 1)), list(range(1, N))], dtype=torch.long)
    rng = np.random.default_rng(1)
    bus_speeds = rng.normal(28, 4, (N, T)).clip(6, 45)
    for s in range(N):                           # a congestion wave that arrives later downstream
        on = 120 + s * 6
        bus_speeds[s, on:on + 30] = rng.normal(9, 2, size=min(30, T - on))
    bus_speeds = np.abs(bus_speeds)

# nodes colored by speed at a mid-window step (same red=slow encoding as METR-LA)
snap_t = min(150, bus_speeds.shape[1] - 1)
sv = bus_speeds[:, snap_t]; vmin, vmax = float(sv.min()), float(sv.max())
def _bcol(v):
    f = 0.0 if vmax == vmin else max(0.0, min(1.0, (v - vmin) / (vmax - vmin)))
    return f"#{int(255*(1-f)):02x}{int(255*f):02x}33"

bm = folium.Map(location=list(np.asarray(bus_latlon).mean(axis=0)), zoom_start=14, tiles=None)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(bm)
folium.TileLayer("CartoDB positron", name="Carto light").add_to(bm)
folium.TileLayer(tiles="", attr="topology-only", name="Blank (topology only)").add_to(bm)
for a, b in bus_edge_index.t().tolist():         # edges = segment-to-segment adjacency
    folium.PolyLine([list(bus_latlon[a]), list(bus_latlon[b])],
                    color="#555", weight=3, opacity=0.7).add_to(bm)
for i, (la, lo_) in enumerate(bus_latlon):       # nodes = segments, colored by speed
    folium.CircleMarker([la, lo_], radius=8, color="black", weight=1,
                        fill=True, fill_color=_bcol(float(sv[i])), fill_opacity=0.9,
                        popup=f"segment node {i} \u00b7 speed {sv[i]:.0f}"
                              + ("" if IS_REAL else " (illustrative)")).add_to(bm)
folium.LayerControl(collapsed=False).add_to(bm)
print(f"bus corridor: {len(bus_latlon)} segment-nodes, {bus_edge_index.shape[1]} edges "
      f"(nodes = segments, edges = adjacency, color = speed)"
      + ("" if IS_REAL else "  [ILLUSTRATIVE Ibn Gabirol stand-in]"))
bm

## (OPTIONAL — trim if short on time) Bus-graph live teaser

A **minimal** "it trains, here's the shape" run — **2–3 epochs only**. NON-SPOILER
by design: **no breakeven, no per-horizon table, no attention interrogation, no
overengineered verdict.** Those are your **practice task**. The demo proves the
machinery runs; you do the analysis.


*What to notice:* the loss **ticks down** — the bus graph trains with the
*same* T-GCN loop as METR-LA. Whether it actually *beats your Unit 3 SARIMA*
(breakeven) is the question you answer in the practice, not here.


In [ ]:
# --- Bus-graph live teaser: 2-3 epochs, NO analysis (non-spoiler) -------------
bus_x = torch.tensor(bus_speeds.T[:, :, None], dtype=torch.float)  # crude [T,nodes,1]
bus_model = TGCNForecaster(node_features=1, hidden=16, horizon=1).to(device)
bus_ei = bus_edge_index.to(device)
opt = torch.optim.Adam(bus_model.parameters(), lr=1e-2)
bus_losses = []
for epoch in range(3):                       # 2-3 epochs ONLY — teaser
    bus_model.train(); ep = 0.0; n = 0
    for t in range(bus_x.shape[0] - 1):
        x = bus_x[t].to(device)              # [nodes, 1]
        y = bus_x[t + 1, :, :1].to(device)   # next-step speed [nodes, 1]
        opt.zero_grad()
        yhat = bus_model(x, bus_ei)
        loss = F.mse_loss(yhat, y)
        loss.backward(); opt.step()
        ep += loss.item(); n += 1
    bus_losses.append(ep / max(n, 1))
    print(f"  bus teaser epoch {epoch+1}/3  loss={bus_losses[-1]:.3f}")
plot_loss_curve(bus_losses, title="Bus-graph T-GCN teaser (it trains — analysis is the practice)")
print("NON-SPOILER: no breakeven / no interrogation here — that is the practice task.")

## Where to go next

Open-ended explorations that lead straight into the supervised-practice arc
(no solutions here — these are yours to run):

1. **Make attention spatial.** Swap A3T-GCN's *temporal* attention for a
   **GAT-style spatial-attention** layer (or inspect GCN message magnitudes) on
   the shockwave: does the "which upstream node" story the model tells **match**
   the cross-correlation evidence? When they disagree, which do you trust?

2. **Interrogate the worst case.** Find the **worst-predicted** sensor/segment in
   your test window and ask what the graph is **missing** — a missing edge? a
   ramp? weather? This negative case is the antidote to attention over-trust.

3. **Push the horizon / go directed.** Re-train with a **directed** adjacency
   (DCRNN's upstream→downstream asymmetry, Li et al. 2018,
   [arXiv:1707.01926](https://arxiv.org/abs/1707.01926)) and see whether the
   breakeven horizon moves — a hand-built preview of why the field went past
   symmetric-GCN T-GCN.


## (OPTIONAL) Appendix A1 — the batched variant: `A3TGCN2`

Reference depth for the PyTorch-fluent subgroup; **fully skippable** (it sits
after "where to go next"). Production traffic code uses the **batched** `A3TGCN2`
— mini-batches over windows — with input shape `[batch, nodes, features, periods]`
(vs. the incremental `A3TGCN(x, edge_index, ...)` per snapshot the main demo
used). **Same math, different plumbing.**

**Checkpoint note:** only **one** METR-LA A3T-GCN checkpoint ships
(`a3tgcn_metrla.pt`, built against `A3TGCN`). This appendix **briefly
live-trains** `A3TGCN2` (2–3 epochs) **purely to show the batched loop runs** — it
ships/loads no separate `a3tgcn2_metrla.pt`. No eval, no breakeven — mechanics
only.


*What to notice:* the batched input is `[batch, nodes, features, periods]`
and the loss ticks down — the *same* model, batched. This is the shape the
practice scaffolding hands you.


In [ ]:
# --- A3TGCN2 batched API: shapes + a tiny 2-3 epoch train (mechanics only) -----
try:
    from torch_geometric_temporal.nn.recurrent import A3TGCN2

    BATCH = 8
    a3t2 = A3TGCN2(in_channels=NUM_FEATURES, out_channels=16,
                   periods=NUM_IN, batch_size=BATCH).to(device)
    # one synthetic batch to show the [batch, nodes, features, periods] shape
    xb = torch.randn(BATCH, NUM_NODES, NUM_FEATURES, NUM_IN).to(device)
    with torch.no_grad():
        out = a3t2(xb, edge_index)
    print(f"A3TGCN2 input  shape: {tuple(xb.shape)}  [batch, nodes, features, periods]")
    print(f"A3TGCN2 output shape: {tuple(out.shape)}")

    opt = torch.optim.Adam(a3t2.parameters(), lr=1e-2)
    app_losses = []
    for epoch in range(3):                    # 2-3 epochs ONLY — mechanics
        xb = torch.randn(BATCH, NUM_NODES, NUM_FEATURES, NUM_IN).to(device)
        yb = torch.randn(BATCH, NUM_NODES, 16).to(device)
        opt.zero_grad()
        loss = F.mse_loss(a3t2(xb, edge_index), yb)
        loss.backward(); opt.step()
        app_losses.append(loss.item())
        print(f"  A3TGCN2 epoch {epoch+1}/3  loss={app_losses[-1]:.3f}")
    print("Appendix: batched A3TGCN2 runs. No eval/breakeven here — mechanics only.")
except ImportError:
    print("A3TGCN2 not available in this PyG-Temporal build — skip the appendix.")